<a href="https://colab.research.google.com/github/rajnarottam38-tech/Starter_Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajnarottam38-tech/Starter_Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane : Ranking Signal analysis is best framed as a scoring problem. The goal is to estimate a continuous score for each content page using observed measurements such as engagement rate, average search position, days since the last update, and trend direction. These measured signals provide evidence about page performance, and the model combines them to estimate an approximate optimization priority rather than the true importance of a page. Pages can then be ranked according to this estimated score. This is not a classification task because the output is a continuous score rather than a fixed category such as "high priority" or "low priority." It is also not a clustering task because clustering groups similar pages without predicting a target or optimization priority. Since the objective is to estimate a continuous priority score from measured signals, this lane is best framed as a scoring problem.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The dataset contains observed measurements such as engagement_rate, avg_position, days_since_last_update, and trend_direction. engagement_rate is used as the target (proxy) because it is the closest available observed measure of page performance. However, it has limitations: an engagement_rate of 0 could indicate either an underperforming page or a newly published page that has not yet accumulated enough engagement. Without additional information, such as page age, this proxy cannot distinguish between the two. In contrast, needs_attention_score is an engineered baseline, not an observed outcome. In a full ML implementation, the model would learn the scoring function from historical data instead of relying on this manually defined rule.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Spearman's Rank Correlation (ρ) is an appropriate success metric because this lane focuses on ranking pages by optimization priority rather than predicting exact values. However, the correlation should be interpreted carefully. The strong correlation between days_since_last_update and needs_attention_score (ρ = 0.848) is expected because days_since_last_update is directly used to construct the engineered score, making this relationship largely circular rather than an independent validation. In contrast, avg_position (ρ = 0.178) was not included in the scoring formula, so its weak positive correlation represents a genuine relationship in the data. Therefore, a defensible evaluation should measure how well the final ranking aligns with independent observed outcomes, rather than relying on correlations with variables used to create the score itself.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/rajnarottam38-tech/Starter_Notebooks/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()
my_lane_columns = [
    "content_id",
    "days_since_last_update",
    "engagement_rate",
    "avg_position",
    "trend_direction",
]

missing = [c for c in my_lane_columns if c not in df.columns]
if missing:
    print("These columns don't exist in df:", missing)

lane_df = df[my_lane_columns].copy()
print("One row = one page, observed at one point in time")
print(f"lane_df shape: {lane_df.shape}")
lane_df.head()
def normalize(series):
    """Scale a column to 0–1 so it's comparable to other signals."""
    return (series - series.min()) / (series.max() - series.min())
engagement_score = 1 - normalize(lane_df["engagement_rate"])
staleness_score = normalize(lane_df["days_since_last_update"])

lane_df["needs_attention_score"] = 0.5 * engagement_score + 0.5 * staleness_score

lane_df.sort_values("needs_attention_score", ascending=False).head(10)

Shape: 30000 rows, 44 columns
One row = one page, observed at one point in time
lane_df shape: (30000, 5)


,content_id,days_since_last_update,engagement_rate,avg_position,trend_direction,needs_attention_score
29384,content_f6fdf87348f6,373,0.0,32.5,down,1.000000
4606,content_3f3576c295f5,373,0.0,1.0,flat,1.000000
26242,content_55a5b1c46474,373,0.0,7.5,down,1.000000
24216,content_1b4ec72dafd4,372,0.0,7.0,down,0.998656
18440,content_8d56efff1e71,372,0.0,35.0,new,0.998656
6962,content_f01216059a6a,335,0.0,5.3,down,0.948925
8631,content_e2b702f4f92b,334,0.0,9.3,down,0.947581
15608,content_06e19c6486b0,334,0.0,5.0,flat,0.947581
21984,content_02b0d6e30129,313,0.0,6.9,down,0.919355
7509,content_7a888d3d99c8,313,0.0,67.6,down,0.919355


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The current needs_attention_score is a manually defined rule that uses only engagement rate and content freshness, so pages with the same values for these two signals receive the same score. However, the dataset shows that such pages can still differ in other important ways. For example, two pages both have 372 days since the last update and 0.0 engagement rate, but one has an average position of 7 with a down trend, while the other has an average position of 35 with a new trend. A fixed rule cannot distinguish between these cases, whereas a machine learning model can learn how multiple signals interact to produce more accurate rankings.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.